In [ ]:
# Install Tiny-SIMA and dependencies into the active Jupyter kernel.
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ.setdefault(
    "XLA_PYTHON_CLIENT_MEM_FRACTION",
    os.environ.get("TINY_SIMA_JAX_MEMORY_FRACTION", "0.60"),
)

if not ((3, 11) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError(
        "Tiny-SIMA requires a Python 3.11 or 3.12 Jupyter kernel."
    )

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the Tiny-SIMA repository.")

has_cuda = (
    shutil.which("nvidia-smi") is not None
    and subprocess.run(
        ["nvidia-smi"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=False,
    ).returncode == 0
)
extras = "notebook,cuda12" if has_cuda else "notebook"
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--quiet",
    "--no-cache-dir",
    "--disable-pip-version-check",
    "-e",
    f"{repo_root}[{extras}]",
])
print("Repository:", repo_root)
print("CUDA detected:", has_cuda)


# Compare learning across environment batch sizes

This final Phase 1 experiment trains 64-, 256-, and 512-environment PPO configurations for the same 1,048,576 environment steps. Each configuration is trained with seeds 0, 1, and 2, then evaluated deterministically for the same 64 episodes.

Runs use fresh subprocesses and save partial results. Rerunning this notebook safely resumes missing training or evaluation jobs.

In [ ]:
ENVIRONMENT_COUNTS = (64, 256, 512)
TRAINING_SEEDS = (0, 1, 2)
TOTAL_TIMESTEPS = 1_048_576
EVALUATION_EPISODES = 64
EVALUATION_SEED = 100
REUSE_EXISTING = True
output_dir = repo_root / "runs" / "learning-scaling-comparison"

print("Training jobs:", len(ENVIRONMENT_COUNTS) * len(TRAINING_SEEDS))
print("Output directory:", output_dir)


In [ ]:
command = [
    sys.executable,
    "-m",
    "tiny_sima.learning_comparison",
    "--config",
    str(repo_root / "configs" / "ppo_1m.yaml"),
    "--num-envs",
    *map(str, ENVIRONMENT_COUNTS),
    "--training-seeds",
    *map(str, TRAINING_SEEDS),
    "--total-timesteps",
    str(TOTAL_TIMESTEPS),
    "--evaluation-episodes",
    str(EVALUATION_EPISODES),
    "--evaluation-seed",
    str(EVALUATION_SEED),
    "--output-dir",
    str(output_dir),
]
if REUSE_EXISTING:
    command.append("--reuse-existing")
subprocess.check_call(command, cwd=repo_root)


In [ ]:
import json

with (output_dir / "comparison_aggregate.json").open() as file:
    aggregate_results = json.load(file)
with (output_dir / "comparison_runs.json").open() as file:
    run_results = json.load(file)

for result in aggregate_results:
    print(
        f"envs={result['num_envs']:>3}  "
        f"return={result['mean_evaluation_return']:.3f} "
        f"+/- {result['evaluation_return_std_across_seeds']:.3f}  "
        f"train_seconds={result['mean_training_seconds']:.1f}  "
        f"updates={result['num_updates']}"
    )
aggregate_results


In [ ]:
import matplotlib.pyplot as plt

environment_counts = [result["num_envs"] for result in aggregate_results]
returns = [result["mean_evaluation_return"] for result in aggregate_results]
return_errors = [
    result["evaluation_return_std_across_seeds"]
    for result in aggregate_results
]
training_seconds = [
    result["mean_training_seconds"] for result in aggregate_results
]

figure, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].errorbar(
    environment_counts,
    returns,
    yerr=return_errors,
    marker="o",
    capsize=5,
)
axes[0].set_xlabel("Parallel environments")
axes[0].set_ylabel("Deterministic evaluation return")
axes[0].set_title("Learning quality")
axes[0].grid(alpha=0.25)

axes[1].plot(environment_counts, training_seconds, marker="o")
axes[1].set_xlabel("Parallel environments")
axes[1].set_ylabel("Mean training seconds")
axes[1].set_title("Wall-clock cost")
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()
